In [1]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.utils import pad_sequences

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)   

In [15]:
df = pd.read_csv("cnn_text_classification.csv")

print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print(df["label"].value_counts())
df.head()
df.info()

Shape: (600, 3)
Missing values: 0
label
1    300
0    300
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      600 non-null    int64
 1   text    600 non-null    str  
 2   label   600 non-null    int64
dtypes: int64(2), str(1)
memory usage: 14.2 KB


In [16]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))

Train: 420 Validation: 90 Test: 90


In [17]:
def tokenize(text):
    return text.lower().split()

train_tokens = [tokenize(t) for t in train_df["text"]]
val_tokens   = [tokenize(t) for t in val_df["text"]]
test_tokens  = [tokenize(t) for t in test_df["text"]]

print(train_df["text"].iloc[0])
print(train_tokens[0])

The movie was frustrating and forgettable
['the', 'movie', 'was', 'frustrating', 'and', 'forgettable']


In [ ]:
vocab = {"<PAD>": 0, "<UNK>": 1}

for sentence in train_tokens: 
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

def to_ids(token_list):
    return [[vocab.get(w, 1) for w in sent] for sent in token_list]

train_ids = to_ids(train_tokens)
val_ids   = to_ids(val_tokens)
test_ids  = to_ids(test_tokens)

print("Vocabulary size:", len(vocab))
print(train_tokens[0])
print(train_ids[0])

Vocabulary size: 79
['the', 'movie', 'was', 'frustrating', 'and', 'forgettable']
[2, 3, 4, 5, 6, 7]


In [19]:
MAXLEN = max(len(s) for s in train_ids)
print("MAXLEN:", MAXLEN)

X_train = pad_sequences(train_ids, maxlen=MAXLEN, padding="post")
X_val   = pad_sequences(val_ids,   maxlen=MAXLEN, padding="post")
X_test  = pad_sequences(test_ids,  maxlen=MAXLEN, padding="post")

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(X_train.shape)
print(X_train[0])

MAXLEN: 9
(420, 9)
[2 3 4 5 6 7 0 0 0]


In [34]:
def build_text_cnn(vocab_size, maxlen, filters=64, kernel_size=3,
                   embedding_dim=32, dropout=0.5):
    inputs = layers.Input(shape=(maxlen,))
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)(inputs)
    x = layers.Conv1D(filters, kernel_size, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    experiment_model = models.Model(inputs, outputs)
    experiment_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return experiment_model


In [51]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.0116 - val_accuracy: 0.9889 - val_loss: 0.0307
Epoch 2/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0083 - val_accuracy: 0.9889 - val_loss: 0.0314
Epoch 3/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0065 - val_accuracy: 0.9889 - val_loss: 0.0320
Epoch 4/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0071 - val_accuracy: 0.9889 - val_loss: 0.0333
Epoch 5/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 0.0066 - val_accuracy: 0.9889 - val_loss: 0.0339
Epoch 6/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 1.0000 - loss: 0.0044 - val_accuracy: 0.9889 - val_loss: 0.0342


In [49]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Test loss: 0.0773
Test accuracy: 0.9667


In [55]:
def predict_sentence(sentence):
    tokens = tokenize(sentence)
    ids = [[vocab.get(w, 1) for w in tokens]]
    x = pad_sequences(ids, maxlen=MAXLEN, padding="post")
    prob = model.predict(x, verbose=0)[0][0]
    label = "positive" if prob >= 0.5 else "negative"
    return prob, label

for s in ["The movie was not good at start but ending was fantastic overall okay",
          "The movie was not to bad"]:
    p, l = predict_sentence(s)
    print(f"{s!r} -> prob={p:.3f} ({l})")

'The movie was not good at start but ending was fantastic overall okay' -> prob=0.467 (negative)
'The movie was not to bad' -> prob=0.769 (positive)
